## Example 1: Basic parameter declaration and validation

In [ ]:
import param

# Example 1: Basic Parameter Declaration and Validation
class BasicExample(param.Parameterized):
    name = param.String(default="example", doc="A string parameter")
    value = param.Number(default=1.0, bounds=(0, 10), doc="A bounded number")
    enabled = param.Boolean(default=True, doc="A boolean flag")

try:
    # Create instance and test parameters
    obj = BasicExample(name="test", value=5.0, enabled=False)
    print(f"Object created: {obj.name}, {obj.value}, {obj.enabled}")
    
    # Parameter validation
    obj.value = 8.5  # Valid
    print(f"Updated value: {obj.value}")
    assert (obj.name, obj.value, obj.enabled) == ("test", 8.5, False)

    # A value outside the declared bounds is rejected
    try:
        obj.value = 11
    except ValueError:
        print("Out-of-bounds value rejected")
    else:
        raise AssertionError("value=11 accepted despite bounds=(0, 10)")
    assert obj.value == 8.5
    
    # Show parameter information
    print(f"Parameters: {list(obj.param.objects().keys())}")
    print(f"Value bounds: {obj.param.value.bounds}")
    
    print("Example 1: Basic parameter declaration completed")
    
except Exception as e:
    print(f"Error in basic parameters: {e}")
    raise AssertionError(
        f"param example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Advanced parameter types and lists

In [ ]:
import param

# Example 2: Advanced Parameter Types and Lists
# (param 2.x: List(class_=...) was removed in favor of item_type; Selector supersedes ObjectSelector)
class AdvancedExample(param.Parameterized):
    choice = param.Selector(default="option1", objects=["option1", "option2", "option3"])
    items = param.List(default=[1, 2, 3], item_type=int, doc="List of integers")
    data = param.Dict(default={'key': 'value'}, doc="Dictionary parameter")
    path = param.Path(default=None, doc="File path parameter")

try:
    # Create instance with advanced parameters
    obj = AdvancedExample()
    print(f"Initial choice: {obj.choice}")
    print(f"Initial items: {obj.items}")
    print(f"Initial data: {obj.data}")
    
    # Test parameter updates
    obj.choice = "option2"
    obj.items = [4, 5, 6, 7]
    obj.data = {'new_key': 'new_value', 'count': 42}
    
    print(f"Updated choice: {obj.choice}")
    print(f"Updated items: {obj.items}")
    print(f"Updated data: {obj.data}")
    
    # Show parameter constraints
    print(f"Choice options: {obj.param.choice.objects}")
    print(f"Items item_type constraint: {obj.param.items.item_type}")
    
    print("Example 2: Advanced parameter types completed")
    
except Exception as e:
    print(f"Error in advanced parameters: {e}")
    raise AssertionError(
        f"param example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Watch and events

In [ ]:
import param

# Example 3: Watch and events
print("Example 3 - Parameter watching")

class WatchableExample(param.Parameterized):
    temperature = param.Number(default=20.0, bounds=(0, 100))
    status = param.String(default="normal")

def temperature_changed(event):
    """Callback for temperature changes"""
    if event.new > 80:
        event.obj.status = "hot"
    elif event.new < 10:
        event.obj.status = "cold"
    else:
        event.obj.status = "normal"
    print(f"Temperature: {event.new}°C -> Status: {event.obj.status}")

sensor = WatchableExample()
print(f"Initial: {sensor.temperature}°C, {sensor.status}")

# Watch parameter changes
sensor.param.watch(temperature_changed, 'temperature')

# Trigger callbacks
sensor.temperature = 85.0
assert sensor.status == "hot"
sensor.temperature = 5.0
sensor.temperature = 25.0

print("Example 3 completed")

## Example 4: Parameter dependencies

In [ ]:
import param

# Example 4: Parameter dependencies
print("Example 4 - Dependencies")

class Rectangle(param.Parameterized):
    width = param.Number(default=10, bounds=(1, 100))
    height = param.Number(default=10, bounds=(1, 100))
    
    @param.depends('width', 'height')
    def area(self):
        """Calculate area from width and height"""
        return self.width * self.height

rect = Rectangle(width=5, height=8)
print(f"Rectangle: {rect.width} x {rect.height}")
print(f"Area: {rect.area()}")

# Update dimensions
rect.width = 12
print(f"Updated width: {rect.width} x {rect.height}")
print(f"New area: {rect.area()}")

rect.height = 6
print(f"Updated height: {rect.width} x {rect.height}")
print(f"New area: {rect.area()}")

print("Example 4 completed")

## Example 5: Serialization and configuration management

In [ ]:
import param
import json

# Example 5: Serialization and Configuration Management
class ConfigExample(param.Parameterized):
    name = param.String(default="config")
    timeout = param.Integer(default=30, bounds=(1, 300))
    debug = param.Boolean(default=False)
    servers = param.List(default=['localhost'], doc="List of server addresses")
    settings = param.Dict(default={'log_level': 'INFO'})

try:
    # Create configuration instance
    config = ConfigExample(
        name="production",
        timeout=60,
        debug=True,
        servers=['server1.com', 'server2.com'],
        settings={'log_level': 'DEBUG', 'max_connections': 100}
    )
    
    print("Configuration created:")
    print(f"Name: {config.name}")
    print(f"Timeout: {config.timeout}s")
    print(f"Debug: {config.debug}")
    print(f"Servers: {config.servers}")
    print(f"Settings: {config.settings}")
    
    # Get parameter values as dict
    param_values = {name: getattr(config, name) for name in config.param.objects().keys()}
    print(f"\nAll parameters: {param_values}")
    
    # Serialize to JSON (simplified)
    serializable = {k: v for k, v in param_values.items() if isinstance(v, (str, int, bool, list, dict))}
    json_str = json.dumps(serializable, indent=2)
    print(f"\nJSON serialization:\n{json_str}")
    
    # Parameter metadata
    print(f"\nParameter documentation:")
    for name, param_obj in config.param.objects('existing').items():
        if param_obj.doc:
            print(f"- {name}: {param_obj.doc}")
    
    print("Example 5: Configuration and serialization completed")
    
except Exception as e:
    print(f"Error in configuration management: {e}")
    raise AssertionError(
        f"param example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e